# How can we anticipate workplace insurance claims?

**MATH4SDG / UDSM — optional, student-led backup project**

**The problem:** An imaginary insurer covers establishments of different sizes, sectors and operating histories. How well can a simple model estimate **the number of claims in a future year**? What additional evidence would be needed before using such a model for decisions?

> **DATA ARE ENTIRELY SYNTHETIC.** No numbers, employers, programme effects or rates in this project come from WCF, TIRA, an insurance company or any actual Tanzanian institution. The generated 'costs' are arbitrary classroom units, **not TZS**. Do not present results as facts about any real sector or country.

All records are produced locally and reproducibly; **no download, credentials or private industry data are required**. The notebook is intended for group work, with optional extensions rather than a fixed teaching timetable. See [project guide](../projects/insurance-claims/README.md).

Code is MIT licensed, educational explanations CC BY 4.0, and the artificial dataset CC0 1.0.

## 1. Define a useful prediction problem

Suppose one establishment covers 20 workers for a full year and another covers 300. Even if their accident risk is identical, which establishment would you expect to report more claims?

We call **worker-years** the total exposure: 50 workers covered for half a year contribute 25 worker-years. The **claims frequency** is

$$\hat\lambda = \frac{\text{number of claims}}{\text{worker-years of exposure}}.$$

We want to estimate the number of claims *expected* in 2025, not claim exactly how many will occur. What information would an actual insurer or WCF have at the **start** of a year? Which columns would only be known afterwards?

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from tools.synthetic_insurance import generate, SYNTHETIC_MARKER
from tools.insurance_methods import (
    aggregate_rate, expected_counts, fit_frequency_model,
    poisson_rate_interval, score_claim_counts, split_by_time, validate,
)

data = generate()  # local, deterministic, ARTIFICIAL; no industry data fetched
validate(data)
print("SYNTHETIC TRAINING DATA ONLY — NOT REAL INSURANCE STATISTICS")
print("Rows:", len(data), "| establishments:", data.establishment_id.nunique(),
      "| years:", sorted(data.year.unique()))
display(data.head())

## 2. Inspect the data rather than trusting a ready-made table

Each row is a *synthetic establishment-year*. The variables `worker_years`, `claims_count`, `sector`, `zone` and `programme` are fictional. `total_claim_cost_units` is determined **after** claims and must never enter a model pretending to predict the year's claims in advance.

Questions: Are there missing sector labels? Are exposures positive? Is the number of claims an integer? Why might repeated years for the same establishment make the observations dependent?

In [ ]:
print("Missing values by column:")
display(data.isna().sum().to_frame("missing_rows"))
assert data["dataset_status"].eq(SYNTHETIC_MARKER).all()
assert (data["worker_years"] > 0).all()
assert (data["claims_count"] >= 0).all()

data["claims_count"].plot.hist(bins=35, figsize=(8, 3), title="Invented claims per establishment-year")
plt.xlabel("Synthetic claim count")
plt.show()

## 3. Compare exposure-adjusted claim rates by sector

**Do not average individual establishment claim frequencies without weighting.** A small establishment with one claim can have an extremely high observed frequency. Aggregate by dividing the **total number of claims** by the **total exposure**.

Does a high total claim count necessarily indicate a high *rate*? Can you think of factors that might explain an observed association, besides the sector itself?

In [ ]:
by_sector = (
    data.assign(sector=data["sector"].fillna("Unreported"))
        .groupby("sector", dropna=False)
        .agg(claims=("claims_count", "sum"),
             worker_years=("worker_years", "sum"),
             rows=("year", "size"))
)
by_sector["claims_per_100_worker_years"] = 100 * by_sector["claims"] / by_sector["worker_years"]
display(by_sector.round(2))
ax = by_sector["claims_per_100_worker_years"].sort_values().plot.barh(
    figsize=(9, 4), title="SYNTHETIC observed rate (not real sector statistics)")
ax.set(xlabel="Claims per 100 worker-years", ylabel="Synthetic sector")
plt.show()

## 4. Evaluation must respect time

Imagine asking, at the beginning of 2025, how many claims will arrive that year. Using actual 2025 claims or costs as predictors would be **data leakage**. A random split of all years can obscure whether the model works in a new year.

We train on **2021–2024** and keep **2025** untouched until final evaluation. The same imaginary establishment can occur in several years. For a truly *new-establishment* prediction problem, a different split and evaluation would be needed.

In [ ]:
train, test = split_by_time(data, test_year=2025)
print("Training years:", sorted(train.year.unique()), "| rows:", len(train))
print("Test years:", sorted(test.year.unique()), "| rows:", len(test))
assert train.year.max() < test.year.min()

## 5. A baseline model you can understand from scratch

If every establishment has the same rate, our simplest predictor is

$$\hat\lambda_{\mathrm{base}} = \frac{\sum_{i\in\mathrm{train}} N_i}{\sum_{i\in\mathrm{train}} e_i},\qquad
\widehat{N}_i = e_i\hat\lambda_{\mathrm{base}},$$

where $e_i$ is worker-year exposure and $N_i$ is observed claims. Why do we multiply the predicted **rate** by exposure when predicting a claim **count**?

The Poisson deviance is one way to score count predictions. It must receive **strictly positive predicted counts**; a smaller score is better. Compare models on the same 2025 rows.

In [ ]:
baseline_rate = aggregate_rate(train)
y_test = test["claims_count"].to_numpy()
exposure_test = test["worker_years"].to_numpy()
baseline_predicted = baseline_rate * exposure_test
base_score = score_claim_counts(y_test, baseline_predicted)
print(f"Historical baseline: {baseline_rate*100:.3f} claims per 100 worker-years")
print(f"Baseline Poisson deviance on 2025: {base_score:.3f}")
print(f"Total 2025 observed claims: {y_test.sum()}; baseline expectation: {baseline_predicted.sum():.1f}")

## 6. A Poisson regression model

One standard count model is

$$N_i\mid x_i,e_i \sim \operatorname{Poisson}(e_i\lambda_i),\qquad
\log(\lambda_i)=\beta_0 + \beta^\top x_i.$$

The *log of exposure* is an **offset** in the count mean, while $x_i$ represents predictors known before the outcome. In scikit-learn, we can equivalently fit the **count per exposure** as the target, weighting rows by their exposure. [Official scikit-learn example](https://scikit-learn.org/stable/auto_examples/linear_model/plot_tweedie_regression_insurance_claims.html).

The predictors below are `sector`, `zone` and `programme`, and **never** the observed claim count or total paid amount. Missing sectors are explicitly labelled "Unreported". What assumptions does this model make? Why might a fitted association with a training programme be **non-causal**?

In [ ]:
model = fit_frequency_model(train)
model_predicted = expected_counts(model, test)
model_score = score_claim_counts(y_test, model_predicted)
print("SYNTHETIC test-year comparison")
display(pd.DataFrame({
    "model": ["Constant rate baseline", "Poisson regression"],
    "mean_Poisson_deviance_2025": [base_score, model_score],
    "expected_2025_claims": [baseline_predicted.sum(), model_predicted.sum()],
    "actual_2025_claims": [y_test.sum(), y_test.sum()],
}).round(2))
print("Models evaluated only on 2025, never fitted on 2025 outcomes.")

## 7. Check where the model works and fails

A model with a lower overall deviance can still systematically underpredict some sectors or a particular year. Compare the *observed* and *expected* rates in 2025 by sector. Why is an aggregate comparison easier to interpret than individual claim-count residuals?

**Warning:** A synthetic sector-level association is not evidence for a real sector-specific risk. Our sample has recurring establishments and unobserved differences in risk.

In [ ]:
comparison = test[["sector", "worker_years", "claims_count"]].copy()
comparison["sector"] = comparison["sector"].fillna("Unreported")
comparison["baseline_count"] = baseline_predicted
comparison["model_count"] = model_predicted
groups = comparison.groupby("sector").agg(
    exposure=("worker_years", "sum"),
    actual=("claims_count", "sum"),
    baseline=("baseline_count", "sum"),
    predicted=("model_count", "sum"),
)
for col in ["actual", "baseline", "predicted"]:
    groups[f"{col}_per_100"] = 100 * groups[col] / groups["exposure"]
display(groups[["exposure", "actual_per_100", "baseline_per_100", "predicted_per_100"]].round(2))
groups[["actual_per_100", "baseline_per_100", "predicted_per_100"]].plot.bar(
    figsize=(10, 4), title="SYNTHETIC 2025 rates: observed vs expected")
plt.ylabel("Claims per 100 worker-years")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 8. Confidence in the rate — and limits of a neat interval

As a starting point, if $K$ independent events follow a Poisson distribution with mean $E\lambda$ for a known exposure $E$, an exact count-based confidence interval can be converted into a rate interval by dividing its endpoints by $E$.

But our imaginary establishments have **persistent unobserved risk differences**, so a simple pooled Poisson interval may understate uncertainty. The data are also repeated over years and conditions change in 2025. Statistical precision conditional on a model does not imply reliable real-world decisions.

In [ ]:
for label, part in [("Historical training period", train), ("Held-out year 2025", test)]:
    claims = int(part.claims_count.sum())
    exposure = float(part.worker_years.sum())
    low, high = poisson_rate_interval(claims, exposure)
    print(f"{label}: {100*claims/exposure:.3f} claims per 100 worker-years; "
          f"Poisson-only 95% interval [{100*low:.3f}, {100*high:.3f}]")
print("Intervals assume a simple Poisson count model; they do not account for establishment clustering.")

## 9. Open the model: why does the baseline work?

Under the simplified assumptions $N_i\sim\operatorname{Poisson}(e_i\lambda)$, the log-likelihood (apart from constants) is

$$\ell(\theta) = \sum_i [N_i\theta - e_i\exp(\theta)],\quad \theta=\log\lambda.$$

Its derivative is $\sum_i (N_i - e_i\exp(\theta))$. Verify with Python that this derivative is approximately zero for the baseline rate computed above.

**AI activity:** Ask an AI assistant to explain the derivative and provide a minimal implementation from scratch. Then independently check shapes, differentiation, the role of exposure, and whether the proposed code accidentally uses 2025 data.

In [ ]:
theta = np.log(baseline_rate)
expected_training = train["worker_years"].to_numpy() * np.exp(theta)
gradient = np.sum(train["claims_count"].to_numpy() - expected_training)
print("Log-likelihood derivative at the fitted baseline:", round(float(gradient), 9))
assert abs(gradient) < 1e-6

## 10. Optional extensions and group presentation

Choose **one** question rather than trying to implement every model:

- **Model checking:** Could an overdispersed or zero-inflated model describe the counts better? Check residuals; do not select models by test-year performance repeatedly.
- **Distribution shift:** Compare training and held-out claim rates. Is the poorer performance in 2025 due to a new risk environment, unobserved factors, or a modelling problem? You cannot determine causality from these columns alone.
- **Claim severity:** The synthetic `total_claim_cost_units` is a heavy-tailed outcome. Study **frequency versus severity** and why predicting total costs is a different problem. Costs are generated only after claims and must not be used as a predictor of those same claims.
- **Validation:** If you want to tune model complexity, tune using earlier years (e.g., 2021–2023 versus 2024) and reserve 2025 for final evaluation.
- **Fairness and practical deployment:** How would you check whether an establishment's apparent risk reflects reporting behavior, data collection practices, or other confounders? Avoid automated decisions affecting workers or establishments from this synthetic exercise.

**Five-minute workshop report:** (1) precise prediction question and units, (2) dataset provenance and its artificial status, (3) baseline and model comparison, (4) one limitation or uncertainty, (5) one question for an industry representative.

**Never upload confidential partner data to this public repository or share them with third-party AI systems without authorization.**